# PyTorch Training Memory

Where does GPU memory go during training and infrence, and how do you avoid `out of memory`?


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

device =  torch.accelerator.current_accelerator()

print("Device:", device)
print("torch:", torch.__version__)


Device: xpu
torch: 2.14.1+xpu


# Memory size
there are some numbers you probably want to remeber for back of the envlope math
1B parameters ~ 0.93 GiB as integers

so since this computer has 18 GiB vram it can never hold 10B parameters in fp16
we will show this now by crashing pytorch

In [2]:
#this would crash on this PC
torch.ones(10**9,10,dtype=torch.float16,device=device)

OutOfMemoryError: XPU out of memory. Tried to allocate 18.63 GiB. GPU 0 has a total capacity of 16.40 GiB of which 16.08 GiB is free. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. Please use `empty_cache` to release all unoccupied cached memory.

In [9]:
#we print here so jupyter doesnt throw a fit and keeps the object
print(torch.ones(10**9,10,dtype=torch.int8,device=device))

tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]], device='xpu:0', dtype=torch.int8)


In [12]:
#this will actually give the memory back to the OS
#its the same as torch.device_name.empty_cache()
getattr(torch, device.type, None).empty_cache()

In [13]:
#CPython usually frees up stuff ASAP but with cycles it sometimes takes it a minute
#in some situation that GPU memory needs to be reused immidatly
#so its good to know this exists
import gc
gc.collect()

26

In [14]:
help(gc.collect)

Help on built-in function collect in module gc:

collect(generation=2)
    Run the garbage collector.

    With no arguments, run a full collection.  The optional argument
    may be an integer specifying which generation to collect.  A ValueError
    is raised if the generation number is invalid.

    The number of unreachable objects is returned.



In [15]:
print(torch.ones(10**9,5,dtype=torch.float16,device=device))
gc.collect()
getattr(torch, device.type, None).empty_cache()

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        ...,
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]], device='xpu:0', dtype=torch.float16)


# Low Level
as you can hopefully see it is important to understand how the low level parts of python work.
there are cases where knowing exactly how the GC works can make the diffrence between the code runing and not runing.

in practice this is USUALLY a non issue, but it is important to be mindfull

In [17]:
BATCH_SIZE = 10**8
FEATURES = 4

f1 = nn.Linear(FEATURES,FEATURES,dtype=torch.float16).to(device)
f2 = nn.Linear(FEATURES,FEATURES,dtype=torch.float16).to(device)
f3 = nn.Linear(FEATURES,FEATURES,dtype=torch.float16).to(device)

#jupyter likes memory leaking objects that are printed automatically
#so we kinda need to fight it and not throw on a big tensor
#its also nice to free up GPU memory  after each run for measurment
backend = getattr(torch, device.type)

import traceback
from IPython.display import display, Code
import gc

def try_run_big(fn):
    try:
        print(fn(torch.ones(BATCH_SIZE,FEATURES,dtype=torch.float16,device=device)))
    except Exception as e:
        print(''.join(traceback.format_exception(e)))
    finally:
        gc.collect()
        backend.empty_cache()


def mem(label):
    backend.synchronize()
    print(
        f"{label}: "
        f"{backend.memory_allocated() / 2**30:.2f} GiB allocated, "
        f"{backend.memory_reserved() / 2**30:.2f} GiB reserved"
    )
        

In [18]:
# #here we will see a lingering x which would make the code crash on memory
@torch.inference_mode()
def bad_func(x):
    mem("input")
    x1 = f1(x)
    mem("after f1")
    x2 = f2(x1)
    mem("after f2")
    x3 = f3(x2)
    mem("after f3")
    return x3

try_run_big(bad_func)

input: 0.75 GiB allocated, 0.75 GiB reserved
after f1: 1.49 GiB allocated, 1.49 GiB reserved
after f2: 2.24 GiB allocated, 2.24 GiB reserved
after f3: 2.98 GiB allocated, 2.99 GiB reserved
tensor([[-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389],
        ...,
        [-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389]], device='xpu:0',
       dtype=torch.float16)


In [19]:
#here we will see that there is less memory used since x is freed up ASAP
@torch.inference_mode()
def good_func(x):
    mem("input")
    x = f1(x)
    mem("after f1")
    x = f2(x)#we overide old x so its memory is freed (thanks to python ref counting)
    mem("after f2")
    x = f3(x)
    mem("after f3")
    return x
    
try_run_big(good_func)

input: 0.75 GiB allocated, 0.75 GiB reserved
after f1: 1.49 GiB allocated, 1.49 GiB reserved
after f2: 1.49 GiB allocated, 2.24 GiB reserved
after f3: 1.49 GiB allocated, 2.24 GiB reserved
tensor([[-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389],
        ...,
        [-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389],
        [-0.1880, -0.2249,  0.0106, -0.2389]], device='xpu:0',
       dtype=torch.float16)


In [24]:
# #here we will see a lingering x which would make the code crash on memory
@torch.inference_mode()
def bad_func(x):
    mem("input")
    history = [x]
    for i in range(100):
        history.append(f1(history[-1]))
        mem(f"iteration {i}")
        
    
    return history[-1]

try_run_big(bad_func)

input: 0.75 GiB allocated, 0.75 GiB reserved
iteration 0: 1.49 GiB allocated, 1.49 GiB reserved
iteration 1: 2.24 GiB allocated, 2.24 GiB reserved
iteration 2: 2.98 GiB allocated, 2.99 GiB reserved
iteration 3: 3.73 GiB allocated, 3.73 GiB reserved
iteration 4: 4.47 GiB allocated, 4.48 GiB reserved
iteration 5: 5.22 GiB allocated, 5.22 GiB reserved
iteration 6: 5.96 GiB allocated, 5.97 GiB reserved
iteration 7: 6.71 GiB allocated, 6.72 GiB reserved
iteration 8: 7.45 GiB allocated, 7.46 GiB reserved
iteration 9: 8.20 GiB allocated, 8.21 GiB reserved
iteration 10: 8.94 GiB allocated, 8.96 GiB reserved
iteration 11: 9.69 GiB allocated, 9.70 GiB reserved
iteration 12: 10.43 GiB allocated, 10.45 GiB reserved
iteration 13: 11.18 GiB allocated, 11.19 GiB reserved
iteration 14: 11.92 GiB allocated, 11.94 GiB reserved
iteration 15: 12.67 GiB allocated, 12.69 GiB reserved
iteration 16: 13.41 GiB allocated, 13.43 GiB reserved
iteration 17: 14.16 GiB allocated, 14.18 GiB reserved
iteration 18: 14.

In [25]:
# #here we will see a lingering x which would make the code crash on memory
@torch.inference_mode()
def good_func(x):
    mem("input")
    for i in range(100):
        x = f1(x)
        mem(f"iteration {i}")
    
    return x

try_run_big(good_func)

input: 0.75 GiB allocated, 0.75 GiB reserved
iteration 0: 1.49 GiB allocated, 1.49 GiB reserved
iteration 1: 1.49 GiB allocated, 2.24 GiB reserved
iteration 2: 1.49 GiB allocated, 2.24 GiB reserved
iteration 3: 1.49 GiB allocated, 2.24 GiB reserved
iteration 4: 1.49 GiB allocated, 2.24 GiB reserved
iteration 5: 1.49 GiB allocated, 2.24 GiB reserved
iteration 6: 1.49 GiB allocated, 2.24 GiB reserved
iteration 7: 1.49 GiB allocated, 2.24 GiB reserved
iteration 8: 1.49 GiB allocated, 2.24 GiB reserved
iteration 9: 1.49 GiB allocated, 2.24 GiB reserved
iteration 10: 1.49 GiB allocated, 2.24 GiB reserved
iteration 11: 1.49 GiB allocated, 2.24 GiB reserved
iteration 12: 1.49 GiB allocated, 2.24 GiB reserved
iteration 13: 1.49 GiB allocated, 2.24 GiB reserved
iteration 14: 1.49 GiB allocated, 2.24 GiB reserved
iteration 15: 1.49 GiB allocated, 2.24 GiB reserved
iteration 16: 1.49 GiB allocated, 2.24 GiB reserved
iteration 17: 1.49 GiB allocated, 2.24 GiB reserved
iteration 18: 1.49 GiB alloca

In [22]:
#if we keep gradients this gets much harder
def grad_func(x):
    mem("input")
    for i in range(100):
        x = f1(x)
        mem(f"iteration {i}")
    
    return x

try_run_big(grad_func)

input: 0.75 GiB allocated, 0.75 GiB reserved
iteration 0: 1.49 GiB allocated, 1.49 GiB reserved
iteration 1: 2.24 GiB allocated, 2.24 GiB reserved
iteration 2: 2.98 GiB allocated, 2.99 GiB reserved
iteration 3: 3.73 GiB allocated, 3.73 GiB reserved
iteration 4: 4.47 GiB allocated, 4.48 GiB reserved
iteration 5: 5.22 GiB allocated, 5.22 GiB reserved
iteration 6: 5.96 GiB allocated, 5.97 GiB reserved
iteration 7: 6.71 GiB allocated, 6.72 GiB reserved
iteration 8: 7.45 GiB allocated, 7.46 GiB reserved
iteration 9: 8.20 GiB allocated, 8.21 GiB reserved
iteration 10: 8.94 GiB allocated, 8.96 GiB reserved
iteration 11: 9.69 GiB allocated, 9.70 GiB reserved
iteration 12: 10.43 GiB allocated, 10.45 GiB reserved
iteration 13: 11.18 GiB allocated, 11.19 GiB reserved
iteration 14: 11.92 GiB allocated, 11.94 GiB reserved
iteration 15: 12.67 GiB allocated, 12.69 GiB reserved
iteration 16: 13.41 GiB allocated, 13.43 GiB reserved
iteration 17: 14.16 GiB allocated, 14.18 GiB reserved
iteration 18: 14.

In [23]:
def bad_grad_func(x):
    mem("input")
    history = [x]
    for i in range(100):
        history.append(f1(history[-1]))
        mem(f"iteration {i}")
        
    
    return history[-1]

try_run_big(bad_grad_func)

input: 0.75 GiB allocated, 0.75 GiB reserved
iteration 0: 1.49 GiB allocated, 1.49 GiB reserved
iteration 1: 2.24 GiB allocated, 2.24 GiB reserved
iteration 2: 2.98 GiB allocated, 2.99 GiB reserved
iteration 3: 3.73 GiB allocated, 3.73 GiB reserved
iteration 4: 4.47 GiB allocated, 4.48 GiB reserved
iteration 5: 5.22 GiB allocated, 5.22 GiB reserved
iteration 6: 5.96 GiB allocated, 5.97 GiB reserved
iteration 7: 6.71 GiB allocated, 6.72 GiB reserved
iteration 8: 7.45 GiB allocated, 7.46 GiB reserved
iteration 9: 8.20 GiB allocated, 8.21 GiB reserved
iteration 10: 8.94 GiB allocated, 8.96 GiB reserved
iteration 11: 9.69 GiB allocated, 9.70 GiB reserved
iteration 12: 10.43 GiB allocated, 10.45 GiB reserved
iteration 13: 11.18 GiB allocated, 11.19 GiB reserved
iteration 14: 11.92 GiB allocated, 11.94 GiB reserved
iteration 15: 12.67 GiB allocated, 12.69 GiB reserved
iteration 16: 13.41 GiB allocated, 13.43 GiB reserved
iteration 17: 14.16 GiB allocated, 14.18 GiB reserved
iteration 18: 14.

**Note** how the gradients are basically forcing us to keep all these intermidate tensors.
This is very important to remeber when runing infrence. using `inference_mode` or `no_grad` will really be a preformance game changer